In [1]:
from data_processing import *
from kernels import *
from methods import *
from utils import *
from features import *
from data_augmentation import *

In [2]:
# Load data
Xtr, Xte, Ytr = load_data(data_dir="data/")
# Preprocess data
std_Xtr, Xtr, Xte = normalize_data(Xtr, Xte)
X_train, Y_train = data_transformations(Xtr, Ytr, only_flip=True)
# Hog features
X_hog_train = extract_hog_features(X_train, cell_size=8, bins=9, block_size=2)
X_hog_test = extract_hog_features(Xte, cell_size=8, bins=9, block_size=2)
# Y labels
Y_train_labels = 2*one_hot_encode(Y_train, 10)-1

Data loaded successfully
Xtr.shape: (5000, 3072)
Xte.shape: (2000, 3072)
Ytr.shape: (5000,)
Before scaling:
Overall mean: 0.00025552399643960884
Overall std: 0.036931775770682074

After scaling:
Xtr mean: 0.006918811541210917
Xtr std: 0.9999999999999999
Xte mean: 0.005958882871338162
Xte std: 0.9983245277147779

Values to save for later use:
mean_Xtr = 0.00025552399643960884
std_Xtr = 0.036931775770682074


In [3]:
# Kernels 
RBF = RBFKernel(gamma=1)
Chi2 = ChiSquareKernel(gamma=0.2)
# Train kernels 
K_train_rbf_hog = RBF(X_hog_train, X_hog_train)
K_train_chi_hog = Chi2(X_hog_train, X_hog_train, block_size=128)
# Test kernels
K_test_rbf_hog = RBF(X_hog_test, X_hog_train)
K_test_chi_hog = Chi2(X_hog_test, X_hog_train, block_size=128)
# For normalization 
K_test_rbf_hog_same = RBF(X_hog_test, X_hog_test)
K_test_chi_hog_same = Chi2(X_hog_test, X_hog_test, block_size=128)

In [4]:
def normalize_kernel(K):
    d = np.sqrt(np.diag(K))
    return K / (d[:,None] * d[None,:] + 1e-12)

def normalize_kernel_val(K, K_val, K_train):
    d_val = np.sqrt(np.diag(K_val))
    d_train = np.sqrt(np.diag(K_train))
    return K / (d_val[:,None] * d_train[None,:] + 1e-12)

In [5]:
# Normalized kernels
K_train_rbf_hog = normalize_kernel(K_train_rbf_hog)
K_train_chi_hog = normalize_kernel(K_train_chi_hog)
K_test_rbf_hog  = normalize_kernel_val(K_test_rbf_hog, K_test_rbf_hog_same, K_train_rbf_hog)
K_test_chi_hog  = normalize_kernel_val(K_test_chi_hog, K_test_chi_hog_same, K_train_chi_hog)
# Final kernels
K_train = 0.25*K_train_rbf_hog + 0.75*K_train_chi_hog
K_test = 0.25*K_test_rbf_hog + 0.75*K_test_chi_hog

In [6]:
model = SVM(K_train)
alphas, b = model.fit(Y_train_labels, verbose=True, progress_every=5)
Y_pred_val = model.predict(K_test, Y_train_labels, alphas, b)
submission_df = format_submission(Y_pred_val)

[SVM.fit] start n=10000 classes=10 max_iter=200 upper_passes=2000
[SVM.fit] class 1/10 started
[SVM.fit] class 1/10 pass 1/200 mode=non-bound changed=4628 non_bound=4236 elapsed=1.1s total=1.2s eta<=2415.3s
[SVM.fit] class 1/10 pass 5/200 mode=non-bound changed=3954 non_bound=3940 elapsed=4.1s total=4.3s eta<=1707.4s
[SVM.fit] class 1/10 pass 10/200 mode=non-bound changed=3911 non_bound=3921 elapsed=7.8s total=8.0s eta<=1588.1s
[SVM.fit] class 1/10 pass 15/200 mode=non-bound changed=3760 non_bound=3921 elapsed=11.5s total=11.6s eta<=1538.6s
[SVM.fit] class 1/10 pass 20/200 mode=non-bound changed=2398 non_bound=3921 elapsed=14.8s total=14.9s eta<=1475.6s
[SVM.fit] class 1/10 pass 25/200 mode=non-bound changed=318 non_bound=3921 elapsed=15.8s total=15.9s eta<=1256.2s
[SVM.fit] class 1/10 pass 30/200 mode=non-bound changed=62 non_bound=3921 elapsed=15.9s total=16.1s eta<=1055.0s
[SVM.fit] class 1/10 pass 35/200 mode=all changed=0 non_bound=3921 elapsed=16.0s total=16.1s eta<=903.9s
[SVM.f

OSError: Cannot save file into a non-existent directory: '/kaggle/working'